# Datasets and DataLoaders
---

## Loading a Dataset
> Here we download a sample dataset.

In [1]:
import torch
from torchvision import datasets
from torch.utils.data import DataLoader

from torchvision.transforms import v2

In [2]:
sample_dataset = datasets.FashionMNIST(
    root="./data",
    train=True, download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(dtype=torch.float32, scale=True)])
)

100%|██████████| 26.4M/26.4M [00:01<00:00, 13.4MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 201kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.75MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 23.8MB/s]


In [3]:
sample_dataset

Dataset FashionMNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
                 ToImage()
                 ToDtype(scale=True)
           )

### Exploring a Dataset

In [4]:
len(sample_dataset)

60000

In [5]:
for X,y in sample_dataset:
  print(type(X), type(y))
  print(X.shape, y)
  break

<class 'torchvision.tv_tensors._image.Image'> <class 'int'>
torch.Size([1, 28, 28]) 9


## Creating Custom Datasets
> A custom dataset must implement the following `__init__`, `__len__` and `__getitem__`

**Note**
This uses the `/content/sample_data` of the google collab environment.

**Goal**
The goal is to create a custom dataset class from existing data.

### EDA

In [6]:
from pathlib import Path
DATA_PATH = Path("/content/sample_data/")
TRAIN_DATA_RAW = DATA_PATH/"california_housing_train.csv"
TEST_DATA_RAW = DATA_PATH/"california_housing_test.csv"

In [25]:
import pandas as pd

train_df = pd.read_csv(TRAIN_DATA_RAW)
test_df = pd.read_csv(TEST_DATA_RAW)
train_df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0
...,...,...,...,...,...,...,...,...,...
16995,-124.26,40.58,52.0,2217.0,394.0,907.0,369.0,2.3571,111400.0
16996,-124.27,40.69,36.0,2349.0,528.0,1194.0,465.0,2.5179,79000.0
16997,-124.30,41.84,17.0,2677.0,531.0,1244.0,456.0,3.0313,103600.0
16998,-124.30,41.80,19.0,2672.0,552.0,1298.0,478.0,1.9797,85800.0


In [26]:
print(train_df.columns)
print(train_df.describe())

Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'median_house_value'],
      dtype='object')
          longitude      latitude  housing_median_age   total_rooms  \
count  17000.000000  17000.000000        17000.000000  17000.000000   
mean    -119.562108     35.625225           28.589353   2643.664412   
std        2.005166      2.137340           12.586937   2179.947071   
min     -124.350000     32.540000            1.000000      2.000000   
25%     -121.790000     33.930000           18.000000   1462.000000   
50%     -118.490000     34.250000           29.000000   2127.000000   
75%     -118.000000     37.720000           37.000000   3151.250000   
max     -114.310000     41.950000           52.000000  37937.000000   

       total_bedrooms    population    households  median_income  \
count    17000.000000  17000.000000  17000.000000   17000.000000   
mean       539.410824   1429.5739

In [27]:
data = train_df.to_numpy()
data[1, 8:9].shape

(1,)

### Define Custom Dataset
> We should create an implementaiton of __len__, __getitem__, and __init__

In [10]:
from torch.utils.data import Dataset, DataLoader

In [141]:
class HousingDataset(Dataset):
  def __init__(self, root: Path, train: bool = True, transform=None, target_transform=None):
    # we load the data already at __init__, we can perform lazy loading here and execute actual loading in getitem if necessary
    print(root/f"california_housing_{'train' if train else 'test'}.csv")
    self.data = pd.read_csv(root/f"california_housing_{'train' if train else 'test'}.csv").to_numpy()
    self.transform = transform
    self.target_transform = target_transform

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    # we can pre-compute this in __init__, if the dataset is small.
    X, y =self. data[idx, :8], self.data[idx, 8:9]
    if self.transform:
      X = self.transform(X)
    if self.target_transform:
      y = self.target_transform(y)
    return X,y


### Transformation
> Scaling and Normalization

In [135]:
data_df = pd.concat([train_df, test_df])
mean, std, eps = data_df.mean(), data_df.std(), 1e-7
X_mean,X_std = mean.to_numpy()[:8], std.to_numpy()[:8]
y_mean,y_std = mean.to_numpy()[8:9], std.to_numpy()[8:9]

# def standard_scale(df: pd.DataFrame, mu, sigma, eps=1-7):

standard_scaler = v2.Lambda(
    lambda X: (X-X_mean)/(X_std+eps)
)

target_standard_scaler = v2.Lambda(
    lambda y: (y-y_mean)/(y_std+eps)
)

X_mean.shape, X_std.shape

((8,), (8,))

In [136]:
# note we can pass on some transform like scaler to perform scaling on the data, for our simple case we dont need to.
train_dataset = HousingDataset(DATA_PATH, train=True, transform=standard_scaler, target_transform=target_standard_scaler)
test_dataset = HousingDataset(DATA_PATH, train=False, transform=standard_scaler, target_transform=target_standard_scaler)


/content/sample_data/california_housing_train.csv
/content/sample_data/california_housing_test.csv


In [137]:
# print(len(train_dataset))
for idx, (X, y) in enumerate(train_dataset):
  print(idx)
  print(X.shape, X, X_mean.shape)
  print(y.shape, y, y_mean.shape)
  break

0
(8,) [ 2.62335224 -0.67259112 -1.08309501  1.36696608  1.77116758 -0.36298855
 -0.07210721 -1.25162442] (8,)
(1,) [-1.21310391] (1,)


## Creating a Custom Dataloader

In [138]:
train_dataloader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)

In [139]:
X,y = next(iter(train_dataloader))
print(X.shape, y.shape)

torch.Size([32, 8]) torch.Size([32, 1])


In [140]:
for X,y in train_dataloader:
  print(X.shape, y.shape)
  break

torch.Size([32, 8]) torch.Size([32, 1])


## TODO
> In the next session, we'll create a `generic neural network predictor` that predicts the median house value given features.